# ALS baseline for popularity calibration

This notebook documents the Spark/Scala environment for the MovieLens 1M recommendation baseline. In the study, the ALS baseline provides the unadjusted recommendations against which fixed and history-aware popularity calibration are compared.

**Current implementation:** library loading, local Spark session initialization, runtime checks, MovieLens rating loading and parsing, DataFrame inspection, full rating and distinct-user counts, rating-value and identifier checks, duplicate-pair detection, per-user history counts, timestamp inspection, a verified per-user training/test split, explicit-feedback ALS training, learned-profile inspection, top-100 candidate generation, per-user relevance scaling, validated top-10 baseline recommendations, verified local Parquet exports, and session shutdown. Prediction quality has not been evaluated.

Run the implemented code cells in order using a Scala 2.12.18 kernel with Java 17. The notebook loads Spark 3.5.9. Saved outputs record a previous execution; rerunning the cells verifies the current environment.


## 1. Verify the local environment

The environment check loads Spark, starts a local session, and counts three synthetic rows. It verifies that the notebook can execute a Spark operation independently of MovieLens data or model training.

The `$ivy` imports load the specified Spark dependencies into the Scala kernel. `spark-sql` provides structured data processing, and `spark-mllib` provides machine-learning algorithms, including ALS. Importing `SparkSession` makes the session entry point available by name.


In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`
import $ivy.`org.apache.spark::spark-mllib:3.5.9`

import org.apache.spark.sql.SparkSession


import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession

### Initialize Spark and verify the runtime

`SparkSession.builder()` configures the application name, local execution with two worker threads, and the driver's loopback addresses. `.getOrCreate()` returns an existing valid session or creates one; `val spark` stores the returned session reference.

After initialization, the log level is set to `ERROR`. The Scala and Java versions are read from the running kernel. `spark.range(3)` creates three synthetic rows, and `.count()` executes the operation and returns their count.

The assertions require Spark 3.5.9, Scala 2.12.18, Java 17, and a row count of three. A failed assertion stops execution of the cell. The printed version numbers and count provide evidence that the environment check completed successfully. These outputs are not recommender evaluation results.


In [2]:
val spark = SparkSession.builder()
  .appName("CS 7265 - ALS Baseline Environment")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val scalaVersion = scala.util.Properties.versionNumberString
val javaVersion = System.getProperty("java.version")
val checkCount = spark.range(3).count()

assert(spark.version == "3.5.9")
assert(scalaVersion == "2.12.18")
assert(System.getProperty("java.specification.version") == "17")
assert(checkCount == 3L)

println(s"Spark: ${spark.version}")
println(s"Scala: $scalaVersion")
println(s"Java: $javaVersion")
println(s"Environment check passed: $checkCount synthetic rows.")


Spark: 3.5.9


Scala: 2.12.18


Java: 17.0.16


Environment check passed: 3 synthetic rows.


spark: SparkSession = org.apache.spark.sql.SparkSession@4a1c9082
scalaVersion: String = "2.12.18"
javaVersion: String = "17.0.16"
checkCount: Long = 3L

## 2. Load MovieLens ratings

The local `data/ml-1m/ratings.dat` file contains one `UserID::MovieID::Rating::Timestamp` record per line. Loading preserves all four fields. The user ID, movie ID, and observed rating are the inputs for explicit-feedback ALS; the timestamp is retained as source information.

### Define the record and parser

`Rating` defines the fields and their numeric types. `RatingParser` is a serializable function from a text line to a `Rating`. Its `apply` method splits a line at `::`, checks for four fields, and converts their values. Keep these definitions in their own cell, separate from the Spark context and data-loading variables, so the parser can be serialized without capturing driver-only notebook state.


In [3]:
case class Rating(userId: Int, movieId: Int, rating: Float, timestamp: Long)

class RatingParser extends (String => Rating) with Serializable {
  def apply(line: String): Rating = {
    val fields = line.split("::")
    require(fields.length == 4, "Expected four fields per MovieLens rating")

    Rating(
      fields(0).toInt,
      fields(1).toInt,
      fields(2).toFloat,
      fields(3).toLong
    )
  }
}


defined class Rating
defined class RatingParser

### Read and parse the local file

The path lookup supports a kernel working directory at either the repository root or its `notebooks` folder. It stops with a descriptive error if the dataset is missing. `sc.textFile` creates an `RDD[String]`, and `map(new RatingParser)` defines the conversion to an `RDD[Rating]`. Spark performs the parsing when an action requests the data. The source file is read without modification.


In [4]:
val ratingsPath = Seq(
  "data/ml-1m/ratings.dat",
  "../data/ml-1m/ratings.dat"
).find(path => new java.io.File(path).isFile).getOrElse {
  throw new IllegalArgumentException("Place MovieLens 1M ratings.dat in data/ml-1m/ and run from the repository root or notebooks folder.")
}

val sc = spark.sparkContext
val lines = sc.textFile(ratingsPath)
val ratingsRDD = lines.map(new RatingParser)


ratingsPath: String = "../data/ml-1m/ratings.dat"
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@4694d992
lines: org.apache.spark.rdd.RDD[String] = ../data/ml-1m/ratings.dat MapPartitionsRDD[8] at textFile at cmd4.sc:9
ratingsRDD: org.apache.spark.rdd.RDD[Rating] = MapPartitionsRDD[9] at map at cmd4.sc:10

### Convert records to a DataFrame

`import spark.implicits._` supplies the conversion helpers for `toDF()`. Spark derives the DataFrame column names and types from `Rating`. The records remain distributed; this conversion does not collect the full dataset into driver memory.


In [5]:
import spark.implicits._
val ratingsDF = ratingsRDD.toDF()


import spark.implicits._
ratingsDF: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 2 more fields]

### Inspect the schema and sample rows

`printSchema()` displays the column structure. `show(5, truncate = false)` executes enough processing to display five records without shortening their contents. This preview checks the loading and conversion of sample rows; it is not a full-dataset validation or a model evaluation. No training split is created here.


In [6]:
ratingsDF.printSchema()
ratingsDF.show(5, truncate = false)


root
 |-- userId: integer (nullable = false)
 |-- movieId: integer (nullable = false)
 |-- rating: float (nullable = false)
 |-- timestamp: long (nullable = false)



+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1193   |5.0   |978300760|
|1     |661    |3.0   |978302109|
|1     |914    |3.0   |978301968|
|1     |3408   |4.0   |978300275|
|1     |2355   |5.0   |978824291|
+------+-------+------+---------+
only showing top 5 rows



## 3. Validate MovieLens ratings

These checks examine the full parsed dataset: record and user counts, rating values, identifier ranges, duplicate user–movie pairs, per-user history counts, and timestamps. They report data-quality issues without modifying the source records. Data validation here is separate from a validation split used to compare model settings.


### Count all ratings and distinct users

`ratingsDF.count()` processes the full parsed dataset and returns the number of rating records. `select("userId").distinct().count()` counts each user ID once without modifying the original DataFrame. Both counts are Spark actions and return `Long` values.

MovieLens 1M documents 1,000,209 ratings from 6,040 users. Matching these totals is a completeness check, but does not establish valid rating ranges, valid identifiers, or unique user–movie pairs. The parser fails on malformed records rather than silently dropping them.


In [7]:
val ratingCount = ratingsDF.count()
val userCount = ratingsDF.select("userId").distinct().count()

println(s"Rating records: $ratingCount")
println(s"Distinct users: $userCount")


Rating records: 1000209


Distinct users: 6040


ratingCount: Long = 1000209L
userCount: Long = 6040L

### Check rating values

MovieLens 1M uses whole-star ratings from 1 through 5. Grouping by `rating` and counting records displays the frequency of each value. The invalid-value check counts nulls and values outside the allowed set, including fractional ratings and non-finite values. It checks the full DataFrame without changing or dropping any source records.

The assertion requires zero invalid ratings. These checks establish rating-value validity; identifier validity and duplicate user–movie pairs require separate checks.


In [8]:
import org.apache.spark.sql.functions.col

ratingsDF.groupBy("rating").count().orderBy("rating").show()

val invalidRatingCount = ratingsDF.filter(
  col("rating").isNull ||
  !col("rating").isin(1.0f, 2.0f, 3.0f, 4.0f, 5.0f)
).count()

println(s"Invalid rating records: $invalidRatingCount")
assert(invalidRatingCount == 0L, "Ratings must be non-null whole-star values from 1 through 5.")


+------+------+
|rating| count|
+------+------+
|   1.0| 56174|
|   2.0|107557|
|   3.0|261197|
|   4.0|348971|
|   5.0|226310|
+------+------+



Invalid rating records: 0


import org.apache.spark.sql.functions.col
invalidRatingCount: Long = 0L

### Check user and movie identifiers

The parser converts both identifiers to `Int`. MovieLens 1M documents user IDs from 1 through 6,040 and movie IDs from 1 through 3,952. The checks require each ID to be non-null and inside its documented inclusive range. The minimum and maximum values summarize the observed endpoints; explicit invalid-record counts check every row.

IDs identify entities; their magnitudes are not preference features. Gaps in movie IDs are allowed, so the largest ID is not a count of distinct movies. These checks do not verify membership in the metadata files or uniqueness of user–movie pairs. Source records remain unchanged.


In [9]:
import org.apache.spark.sql.functions.{min, max}

ratingsDF.agg(
  min("userId").as("minUserId"),
  max("userId").as("maxUserId"),
  min("movieId").as("minMovieId"),
  max("movieId").as("maxMovieId")
).show()

val invalidUserIdCount = ratingsDF.filter(
  col("userId").isNull || !col("userId").between(1, 6040)
).count()

val invalidMovieIdCount = ratingsDF.filter(
  col("movieId").isNull || !col("movieId").between(1, 3952)
).count()

println(s"Records with invalid user IDs: $invalidUserIdCount")
println(s"Records with invalid movie IDs: $invalidMovieIdCount")

assert(invalidUserIdCount == 0L, "User IDs must be non-null integers from 1 through 6040.")
assert(invalidMovieIdCount == 0L, "Movie IDs must be non-null integers from 1 through 3952.")


+---------+---------+----------+----------+
|minUserId|maxUserId|minMovieId|maxMovieId|
+---------+---------+----------+----------+
|        1|     6040|         1|      3952|
+---------+---------+----------+----------+



Records with invalid user IDs: 0


Records with invalid movie IDs: 0


import org.apache.spark.sql.functions.{min, max}
invalidUserIdCount: Long = 0L
invalidMovieIdCount: Long = 0L

### Check duplicate user–movie pairs

Grouping by both `userId` and `movieId` counts observations for each interaction. Groups with more than one record are duplicate pairs, even if their ratings or timestamps differ. `duplicatePairCount` counts the affected pairs, not the number of extra rows.

The assertion requires one observation per user–movie pair in the source dataset. Duplicate observations could give interactions extra weight during training, and conflicting ratings would require an explicit resolution policy. This check reports the issue without removing records.


In [10]:
val duplicatePairs = ratingsDF
  .groupBy("userId", "movieId")
  .count()
  .filter(col("count") > 1L)

val duplicatePairCount = duplicatePairs.count()
println(s"Repeated user-movie pairs: $duplicatePairCount")

assert(duplicatePairCount == 0L, "Each user-movie pair must appear only once.")


Repeated user-movie pairs: 0


duplicatePairs: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 1 more field]
duplicatePairCount: Long = 0L

### Check the number of ratings per user

Grouping by `userId` counts the observations contributed by each user. The grouped count is renamed `ratingCount` to distinguish a user's history length from an individual star rating. The minimum and maximum summarize the observed history lengths, and the assertion checks MovieLens 1M's documented minimum of 20 ratings per user in the complete source dataset.

This source-data check does not remove users or define history groups for an experiment. Training-only history counts must be recomputed from the agreed training split; they may be smaller than these full-dataset counts.


In [11]:
val ratingsPerUser = ratingsDF
  .groupBy("userId")
  .count()
  .withColumnRenamed("count", "ratingCount")

ratingsPerUser.agg(
  min("ratingCount").as("minRatingsPerUser"),
  max("ratingCount").as("maxRatingsPerUser")
).show()

val usersBelowMinimum = ratingsPerUser
  .filter(col("ratingCount") < 20L)
  .count()

println(s"Users with fewer than 20 ratings: $usersBelowMinimum")
assert(usersBelowMinimum == 0L, "The complete MovieLens 1M dataset must contain at least 20 ratings per user.")


+-----------------+-----------------+
|minRatingsPerUser|maxRatingsPerUser|
+-----------------+-----------------+
|               20|             2314|
+-----------------+-----------------+



Users with fewer than 20 ratings: 0


ratingsPerUser: org.apache.spark.sql.package.DataFrame = [userId: int, ratingCount: bigint]
usersBelowMinimum: Long = 0L

### Inspect source timestamps

MovieLens stores Unix timestamps in seconds. The parser requires `Long` values; this additional check rejects null and nonpositive timestamps for this post-1970 dataset. The minimum and maximum timestamps are converted to UTC dates for inspection. These basic checks do not verify the correctness of every recorded date.

The aggregation produces one summary row. `first()` retrieves that row, `getAs[Long]` reads a named field, and `Instant.ofEpochSecond` converts it to a UTC instant. The original timestamp column is unchanged. Timestamps are not ALS features, and this inspection does not define a temporal split.


In [12]:
import java.time.Instant

val invalidTimestampCount = ratingsDF.filter(
  col("timestamp").isNull || col("timestamp") <= 0L
).count()

println(s"Records with missing or nonpositive timestamps: $invalidTimestampCount")
assert(invalidTimestampCount == 0L, "MovieLens timestamps must be present and positive Unix seconds.")

val timestampBounds = ratingsDF.agg(
  min("timestamp").as("earliestTimestamp"),
  max("timestamp").as("latestTimestamp")
).first()

val earliestRating = Instant.ofEpochSecond(timestampBounds.getAs[Long]("earliestTimestamp"))
val latestRating = Instant.ofEpochSecond(timestampBounds.getAs[Long]("latestTimestamp"))

println(s"Earliest rating (UTC): $earliestRating")
println(s"Latest rating (UTC): $latestRating")


Records with missing or nonpositive timestamps: 0


Earliest rating (UTC): 2000-04-25T23:05:32Z


Latest rating (UTC): 2003-02-28T17:49:50Z


import java.time.Instant
invalidTimestampCount: Long = 0L
timestampBounds: org.apache.spark.sql.Row = [956703932,1046454590]
earliestRating: Instant = 2000-04-25T23:05:32Z
latestRating: Instant = 2003-02-28T17:49:50Z

## 4. Create the per-user training/test split

Each user contributes `floor(0.8 * n)` of their `n` ratings to training and the remaining ratings to test. This preserves training and held-out observations for every user in the validated MovieLens 1M dataset. The per-user design is informed by the [LensKit splitting guide](https://lenskit.org/stable/guide/splitting#user-based-splitting), which describes controlling experimental conditions for individual users. The 80/20 proportion comes from the proposal; seed 42, rounding, and the hash-based ordering are project implementation choices.

### Set the allocation and count each user's rows

An unordered window groups calculations by user while retaining each rating record. `userRatingCount` contains the full source-history length used to allocate the split. Experimental history counts must instead be calculated from the resulting training data.


In [13]:
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions.{count, lit, xxhash64, row_number, floor}

val trainingFraction = 0.8
val splitSeed = 42L

val perUser = Window.partitionBy("userId")
val ratingsWithCounts = ratingsDF.withColumn(
  "userRatingCount",
  count("*").over(perUser)
)


import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions.{count, lit, xxhash64, row_number, floor}
trainingFraction: Double = 0.8
splitSeed: Long = 42L
perUser: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@42f1b07d
ratingsWithCounts: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 3 more fields]

### Assign a reproducible shuffle order

`xxhash64` creates a deterministic pseudo-random ordering key from the split seed and the user–movie identifiers. The split seed is an input column to the hash. Movie ID breaks hash ties within each user. `row_number` numbers positions starting at one separately for each user.


In [14]:
val ratingsWithKeys = ratingsWithCounts.withColumn(
  "shuffleKey",
  xxhash64(lit(splitSeed), col("userId"), col("movieId"))
)

val shuffledPerUser = Window
  .partitionBy("userId")
  .orderBy(col("shuffleKey"), col("movieId"))

val rankedRatings = ratingsWithKeys.withColumn(
  "ratingPosition",
  row_number().over(shuffledPerUser)
)


ratingsWithKeys: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 4 more fields]
shuffledPerUser: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@4b4c8598
rankedRatings: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 5 more fields]

### Assign rows to training and test

Positions at or below a user's rounded-down training count belong to training. All remaining positions belong to test. Selecting the original four columns removes temporary split-construction fields from the model inputs. The source DataFrame remains unchanged.


In [15]:
val ratingsWithCutoff = rankedRatings.withColumn(
  "trainingCount",
  floor(col("userRatingCount") * trainingFraction)
)

val training = ratingsWithCutoff.filter(
  col("ratingPosition") <= col("trainingCount")
)
val test = ratingsWithCutoff.filter(
  col("ratingPosition") > col("trainingCount")
)

val trainingRatings = training.select("userId", "movieId", "rating", "timestamp")
val testRatings = test.select("userId", "movieId", "rating", "timestamp")


ratingsWithCutoff: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 6 more fields]
training: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 6 more fields]
test: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 6 more fields]
trainingRatings: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 2 more fields]
testRatings: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 2 more fields]

### Verify separation, reconstruction, and per-user allocation

The overlap check compares user–movie keys. Combining both sets and using `exceptAll` in both directions checks that all original records. These operations compare columns by position; all three DataFrames use the original column order.

Expected training counts are computed from the original per-user counts and compared with actual training counts. Once separation and exact reconstruction pass, matching each user's training count also establishes the complementary test count. All failure counts must be zero. Training/test record and user counts summarize the resulting allocation; rounded per-user cutoffs make the aggregate proportion approximate.


In [16]:
val overlapCount = trainingRatings
  .select("userId", "movieId")
  .intersect(testRatings.select("userId", "movieId"))
  .count()

val combinedRatings = trainingRatings.union(testRatings)
val missingRecordCount = ratingsDF.exceptAll(combinedRatings).count()
val unexpectedRecordCount = combinedRatings.exceptAll(ratingsDF).count()

val expectedTrainingCounts = ratingsPerUser.select(
  col("userId"),
  floor(col("ratingCount") * trainingFraction).as("count")
)
val actualTrainingCounts = trainingRatings.groupBy("userId").count()
val mismatchedUserCount = expectedTrainingCounts.exceptAll(actualTrainingCounts).count()

println(s"Overlapping user-movie pairs: $overlapCount")
println(s"Missing original records: $missingRecordCount")
println(s"Unexpected or excess records: $unexpectedRecordCount")
println(s"Users with incorrect training allocation: $mismatchedUserCount")

assert(overlapCount == 0L, "Training and test must not overlap.")
assert(missingRecordCount == 0L, "The split must preserve every original record.")
assert(unexpectedRecordCount == 0L, "The split must not add or duplicate records.")
assert(mismatchedUserCount == 0L, "Every user must receive the intended training allocation.")

val trainingRatingCount = trainingRatings.count()
val testRatingCount = testRatings.count()
val trainingUserCount = trainingRatings.select("userId").distinct().count()
val testUserCount = testRatings.select("userId").distinct().count()

println(s"Training: $trainingRatingCount ratings from $trainingUserCount users")
println(s"Test: $testRatingCount ratings from $testUserCount users")


Overlapping user-movie pairs: 0


Missing original records: 0


Unexpected or excess records: 0


Users with incorrect training allocation: 0


Training: 797758 ratings from 6040 users


Test: 202451 ratings from 6040 users


overlapCount: Long = 0L
combinedRatings: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
missingRecordCount: Long = 0L
unexpectedRecordCount: Long = 0L
expectedTrainingCounts: org.apache.spark.sql.package.DataFrame = [userId: int, count: bigint]
actualTrainingCounts: org.apache.spark.sql.package.DataFrame = [userId: int, count: bigint]
mismatchedUserCount: Long = 0L
trainingRatingCount: Long = 797758L
testRatingCount: Long = 202451L
trainingUserCount: Long = 6040L
testUserCount: Long = 6040L

## 5. ALS baseline

The baseline uses explicit-feedback ALS: observed star ratings are the values the model learns to predict. The project proposal specifies rank 50, 10 iterations, and regularization 0.1. These are fixed baseline settings; no validation-based parameter tuning is performed here.

`splitSeed` controls the training/test allocation, while `alsSeed` controls the initial factor values used by ALS. Both are named separately and set to `42L`, a Scala `Long` literal.


In [17]:
import org.apache.spark.ml.recommendation.ALS

val alsSeed = 42L

val als = new ALS()
  .setUserCol("userId")
  .setItemCol("movieId")
  .setRatingCol("rating")
  .setImplicitPrefs(false)
  .setRank(50)
  .setMaxIter(10)
  .setRegParam(0.1)
  .setSeed(alsSeed)


import org.apache.spark.ml.recommendation.ALS
alsSeed: Long = 42L
als: ALS = als_a8fce24448f6

### Train on the training partition

Calling `fit(trainingRatings)` runs ALS to learn one numerical profile for each user and movie represented in training. It alternates between updating user and movie profiles using observed ratings and regularization. The timestamp column is retained in the input DataFrame but is not a model input.

The returned `ALSModel` contains the learned profiles. Test ratings are not used during training. Training alone does not generate recommendation lists or measure prediction quality.


In [18]:
val model = als.fit(trainingRatings)

println(s"ALS training complete: rank ${model.rank}, ${als.getMaxIter} iterations, regularization ${als.getRegParam}, seed ${als.getSeed}.")


ALS training complete: rank 50, 10 iterations, regularization 0.1, seed 42.


model: org.apache.spark.ml.recommendation.ALSModel = ALSModel: uid=als_a8fce24448f6, rank=50

### Inspect the learned profiles

`userFactors` and `itemFactors` are DataFrames with an `id` column and a `features` array. The IDs correspond to the original user and movie IDs. Each array contains 50 learned factor values; the coordinates are not named genres or observed ratings. A user profile and a movie profile produce a predicted rating through their dot product.

The checks below verify that the model contains a profile for every training user and movie and that every profile has the configured number of factors. The previews display only the first five values of each 50-value profile. These checks establish the model's structure. A movie absent from training has no learned profile.

The model remains in the Spark session; these cells save no model files. Saved notebook outputs preserve the inspection results after the session is stopped.


In [19]:
import org.apache.spark.sql.functions.{size, slice}

val learnedUserCount = model.userFactors.count()
val learnedMovieCount = model.itemFactors.count()
val trainingMovieCount = trainingRatings.select("movieId").distinct().count()

assert(model.rank == als.getRank, "The model rank must match the ALS configuration.")
assert(learnedUserCount == trainingUserCount, "Every training user must have a learned profile.")
assert(learnedMovieCount == trainingMovieCount, "Every training movie must have a learned profile.")
assert(model.userFactors.filter(size(col("features")) =!= model.rank).count() == 0L,
  "Every user profile must contain the configured number of factors.")
assert(model.itemFactors.filter(size(col("features")) =!= model.rank).count() == 0L,
  "Every movie profile must contain the configured number of factors.")

println(s"Learned user profiles: $learnedUserCount")
println(s"Learned movie profiles: $learnedMovieCount")
println(s"Factors per profile: ${model.rank}")

println("User profile schema and preview:")
model.userFactors.printSchema()
model.userFactors.orderBy("id").select(
  col("id").as("userId"),
  size(col("features")).as("factorCount"),
  slice(col("features"), 1, 5).as("firstFiveFeatures")
).show(3, truncate = false)

println("Movie profile schema and preview:")
model.itemFactors.printSchema()
model.itemFactors.orderBy("id").select(
  col("id").as("movieId"),
  size(col("features")).as("factorCount"),
  slice(col("features"), 1, 5).as("firstFiveFeatures")
).show(3, truncate = false)


Learned user profiles: 6040


Learned movie profiles: 3685


Factors per profile: 50


User profile schema and preview:


root
 |-- id: integer (nullable = false)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = false)



+------+-----------+-----------------------------------------------------------------+
|userId|factorCount|firstFiveFeatures                                                |
+------+-----------+-----------------------------------------------------------------+
|1     |50         |[-0.5176357, 0.018918697, -0.2998146, -0.4567224, -0.47019693]   |
|2     |50         |[-0.4901694, -0.097104445, -0.3915142, -0.3053655, -0.28436658]  |
|3     |50         |[-0.48072392, -0.12430852, -0.18470964, -0.53348285, -0.17757891]|
+------+-----------+-----------------------------------------------------------------+
only showing top 3 rows



Movie profile schema and preview:


root
 |-- id: integer (nullable = false)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = false)



+-------+-----------+------------------------------------------------------------------+
|movieId|factorCount|firstFiveFeatures                                                 |
+-------+-----------+------------------------------------------------------------------+
|1      |50         |[-0.3073223, 0.058646027, -0.27411053, -0.45746616, -0.24898122]  |
|2      |50         |[-0.20004453, -0.077421986, -0.10877516, -0.26099002, -0.07944111]|
|3      |50         |[-0.31387538, -0.08049874, 0.18931799, -0.17542289, -0.18691266]  |
+-------+-----------+------------------------------------------------------------------+
only showing top 3 rows



import org.apache.spark.sql.functions.{size, slice}
learnedUserCount: Long = 6040L
learnedMovieCount: Long = 3685L
trainingMovieCount: Long = 3685L

## 6. Generate baseline recommendations

The proposal specifies 100 candidate movies per user and a top-10 unadjusted recommendation list. Eligibility is defined relative to the training history: user–movie pairs in `trainingRatings` are excluded, while held-out test pairs remain eligible for later evaluation. Only users and movies with learned profiles are scored.

The cross join constructs all known user–movie pairs. A left anti join removes pairs already present in training, matching both identifiers.


In [20]:
val candidatePoolSize = 100
val recommendationListSize = 10

val knownUsers = model.userFactors.select(col("id").as("userId"))
val knownMovies = model.itemFactors.select(col("id").as("movieId"))
val possiblePairs = knownUsers.crossJoin(knownMovies)
val ratedPairs = trainingRatings.select("userId", "movieId")

val candidatePairs = possiblePairs.join(
  ratedPairs,
  Seq("userId", "movieId"),
  "left_anti"
)


candidatePoolSize: Int = 100
recommendationListSize: Int = 10
knownUsers: org.apache.spark.sql.package.DataFrame = [userId: int]
knownMovies: org.apache.spark.sql.package.DataFrame = [movieId: int]
possiblePairs: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int]
ratedPairs: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int]
candidatePairs: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int]

### Score eligible pairs

`model.transform` adds the predicted rating calculated from each user's and movie's learned profiles. The profiles remain fixed. Predictions are relevance estimates and may fall outside the observed 1–5 rating range.

The full scored DataFrame is persisted on local disk because generation and validation reuse it. Persistence is temporary Spark storage, not an exported dataset. The scoring checks require one finite prediction per eligible pair before ranking begins.


In [21]:
import org.apache.spark.storage.StorageLevel
import org.apache.spark.sql.functions.{when, isnan, abs, sum, countDistinct, lag}

val scoredCandidates = model.transform(candidatePairs)
  .persist(StorageLevel.DISK_ONLY)

val scoringSummary = scoredCandidates.agg(
  count("*").as("scoredCount"),
  sum(when(
    col("prediction").isNull || isnan(col("prediction")) ||
    abs(col("prediction")) === lit(Double.PositiveInfinity),
    1L
  ).otherwise(0L)).as("invalidPredictionCount")
).first()

val scoredCandidateCount = scoringSummary.getAs[Long]("scoredCount")
val invalidPredictionCount = scoringSummary.getAs[Long]("invalidPredictionCount")
val expectedCandidateCount = learnedUserCount * learnedMovieCount - trainingRatingCount

println(s"Eligible pairs scored: $scoredCandidateCount")
println(s"Missing, NaN, or infinite predictions: $invalidPredictionCount")
assert(scoredCandidateCount == expectedCandidateCount,
  "Every eligible user-movie pair must receive a prediction.")
assert(invalidPredictionCount == 0L, "All candidate predictions must be finite.")


Eligible pairs scored: 21459642


Missing, NaN, or infinite predictions: 0


import org.apache.spark.storage.StorageLevel
import org.apache.spark.sql.functions.{when, isnan, abs, sum, countDistinct, lag}
scoredCandidates: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 1 more field]
scoringSummary: org.apache.spark.sql.Row = [21459642,0]
scoredCandidateCount: Long = 21459642L
invalidPredictionCount: Long = 0L
expectedCandidateCount: Long = 21459642L

### Rank and retain the top 100

Each user's movies are ordered by descending predicted rating, with ascending movie ID breaking ties consistently. `recommendationRank` is a position in the list, distinct from the ALS model's factor rank. The filter retains positions 1–100 separately for each user.


In [22]:
val recommendationOrder = Window
  .partitionBy("userId")
  .orderBy(col("prediction").desc, col("movieId").asc)

val rankedCandidates = scoredCandidates.withColumn(
  "recommendationRank",
  row_number().over(recommendationOrder)
)

val topCandidates = rankedCandidates.filter(
  col("recommendationRank") <= candidatePoolSize
)


recommendationOrder: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@59f82b47
rankedCandidates: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 2 more fields]
topCandidates: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]

### Scale relevance and select the baseline list

Min–max scaling is applied separately within each user's top-100 candidate pool: `(prediction - minimum) / (maximum - minimum)`. It preserves the ranking and supplies relevance scores in the 0–1 range for subsequent calibration. Zero means the lowest prediction within this pool.

If all predictions in a user's pool are equal, every scaled score is set to 1.0 to preserve their equal standing without dividing by zero. Calculations use double precision, while the original prediction is retained. The baseline takes ranks 1–10 and retains both score columns. The smaller scaled pool is persisted in memory with disk fallback for repeated checks and previews.


In [23]:
val candidateWindow = Window.partitionBy("userId")

val candidatesWithBounds = topCandidates
  .withColumn("minPrediction", min(col("prediction")).over(candidateWindow))
  .withColumn("maxPrediction", max(col("prediction")).over(candidateWindow))

val scaledCandidates = candidatesWithBounds
  .withColumn(
    "relevanceScore",
    when(col("maxPrediction") === col("minPrediction"), lit(1.0))
      .otherwise(
        (col("prediction").cast("double") - col("minPrediction").cast("double")) /
        (col("maxPrediction").cast("double") - col("minPrediction").cast("double"))
      )
  )
  .drop("minPrediction", "maxPrediction")
  .persist(StorageLevel.MEMORY_AND_DISK)

val baselineRecommendations = scaledCandidates.filter(
  col("recommendationRank") <= recommendationListSize
)


candidateWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@504d8994
candidatesWithBounds: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 4 more fields]
scaledCandidates: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 3 more fields]
baselineRecommendations: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 3 more fields]

## 7. Validate recommendation lists

These checks establish structural correctness. Every training user must receive 100 distinct candidates and 10 baseline recommendations. The user-ID comparisons detect missing users even when they have no output rows; row counts alone cannot establish coverage.


In [24]:
val expectedRecommendationUsers = trainingRatings.select("userId").distinct()
val candidateUsers = scaledCandidates.select("userId").distinct()
val baselineUsers = baselineRecommendations.select("userId").distinct()

val missingCandidateUsers = expectedRecommendationUsers.except(candidateUsers).count()
val unexpectedCandidateUsers = candidateUsers.except(expectedRecommendationUsers).count()
val missingBaselineUsers = expectedRecommendationUsers.except(baselineUsers).count()
val unexpectedBaselineUsers = baselineUsers.except(expectedRecommendationUsers).count()

val candidateListSummary = scaledCandidates.groupBy("userId").agg(
  count("*").as("listSize"),
  countDistinct("movieId").as("distinctMovies"),
  countDistinct("recommendationRank").as("distinctRanks"),
  min("recommendationRank").as("firstRank"),
  max("recommendationRank").as("lastRank")
)
val invalidCandidateLists = candidateListSummary.filter(
  col("listSize") =!= candidatePoolSize ||
  col("distinctMovies") =!= candidatePoolSize ||
  col("distinctRanks") =!= candidatePoolSize ||
  col("firstRank") =!= 1 || col("lastRank") =!= candidatePoolSize
).count()

val baselineListSummary = baselineRecommendations.groupBy("userId").agg(
  count("*").as("listSize"),
  countDistinct("movieId").as("distinctMovies"),
  countDistinct("recommendationRank").as("distinctRanks"),
  min("recommendationRank").as("firstRank"),
  max("recommendationRank").as("lastRank")
)
val invalidBaselineLists = baselineListSummary.filter(
  col("listSize") =!= recommendationListSize ||
  col("distinctMovies") =!= recommendationListSize ||
  col("distinctRanks") =!= recommendationListSize ||
  col("firstRank") =!= 1 || col("lastRank") =!= recommendationListSize
).count()

println(s"Candidate users missing/unexpected: $missingCandidateUsers / $unexpectedCandidateUsers")
println(s"Baseline users missing/unexpected: $missingBaselineUsers / $unexpectedBaselineUsers")
println(s"Invalid candidate lists (size, uniqueness, ranks): $invalidCandidateLists")
println(s"Invalid baseline lists (size, uniqueness, ranks): $invalidBaselineLists")
assert(missingCandidateUsers == 0L && unexpectedCandidateUsers == 0L,
  "Candidate lists must cover exactly the training users.")
assert(missingBaselineUsers == 0L && unexpectedBaselineUsers == 0L,
  "Baseline lists must cover exactly the training users.")
assert(invalidCandidateLists == 0L, "Each user must have 100 distinct candidates at ranks 1-100.")
assert(invalidBaselineLists == 0L, "Each user must have 10 distinct baseline movies at ranks 1-10.")


Candidate users missing/unexpected: 0 / 0


Baseline users missing/unexpected: 0 / 0


Invalid candidate lists (size, uniqueness, ranks): 0


Invalid baseline lists (size, uniqueness, ranks): 0


expectedRecommendationUsers: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int]
candidateUsers: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int]
baselineUsers: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int]
missingCandidateUsers: Long = 0L
unexpectedCandidateUsers: Long = 0L
missingBaselineUsers: Long = 0L
unexpectedBaselineUsers: Long = 0L
candidateListSummary: org.apache.spark.sql.package.DataFrame = [userId: int, listSize: bigint ... 4 more fields]
invalidCandidateLists: Long = 0L
baselineListSummary: org.apache.spark.sql.package.DataFrame = [userId: int, listSize: bigint ... 4 more fields]
invalidBaselineLists: Long = 0L

### Check exclusions and ranking

No candidate pair may appear in training. Within each candidate list, adjacent rows must follow descending prediction and ascending movie ID for ties. Finally, every eligible movie omitted from the top 100 is compared with that user's rank-100 cutoff: none may outrank it. This checks selection against the full scored set.


In [25]:
val recommendedTrainingPairs = scaledCandidates
  .join(ratedPairs, Seq("userId", "movieId"), "inner").count()

val rankOrder = Window.partitionBy("userId").orderBy("recommendationRank")
val adjacentCandidates = scaledCandidates
  .withColumn("previousPrediction", lag(col("prediction"), 1).over(rankOrder))
  .withColumn("previousMovieId", lag(col("movieId"), 1).over(rankOrder))
val outOfOrderCandidates = adjacentCandidates.filter(
  col("recommendationRank") > 1 && (
    col("prediction") > col("previousPrediction") ||
    (col("prediction") === col("previousPrediction") &&
      col("movieId") < col("previousMovieId"))
  )
).count()

val candidateCutoffs = scaledCandidates
  .filter(col("recommendationRank") === candidatePoolSize)
  .select(
    col("userId"),
    col("prediction").as("cutoffPrediction"),
    col("movieId").as("cutoffMovieId")
  )
val omittedCandidates = scoredCandidates.join(
  scaledCandidates.select("userId", "movieId"),
  Seq("userId", "movieId"),
  "left_anti"
)
val omittedBetterCandidates = omittedCandidates
  .join(candidateCutoffs, Seq("userId"), "inner")
  .filter(
    col("prediction") > col("cutoffPrediction") ||
    (col("prediction") === col("cutoffPrediction") &&
      col("movieId") < col("cutoffMovieId"))
  ).count()

println(s"Candidate pairs found in training: $recommendedTrainingPairs")
println(s"Candidates out of rank order: $outOfOrderCandidates")
println(s"Omitted candidates that outrank the cutoff: $omittedBetterCandidates")
assert(recommendedTrainingPairs == 0L, "Training-rated pairs must be excluded.")
assert(outOfOrderCandidates == 0L, "Candidate ranks must follow the score and tie-break order.")
assert(omittedBetterCandidates == 0L, "No omitted eligible movie may outrank the top-100 cutoff.")


Candidate pairs found in training: 0


Candidates out of rank order: 0


Omitted candidates that outrank the cutoff: 0


recommendedTrainingPairs: Long = 0L
rankOrder: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@5c21d28
adjacentCandidates: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 5 more fields]
outOfOrderCandidates: Long = 0L
candidateCutoffs: org.apache.spark.sql.package.DataFrame = [userId: int, cutoffPrediction: float ... 1 more field]
omittedCandidates: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 1 more field]
omittedBetterCandidates: Long = 0L

### Check scaled scores and baseline membership

Scaled scores must be finite and within 0–1. For pools with differing predictions, the endpoints must be 0 and 1; constant-score pools must follow the all-ones convention. A small tolerance accommodates floating-point arithmetic. The baseline is compared in both directions with the candidate pool's first 10 rows, including predictions, ranks, and scaled scores.


In [26]:
val scoreTolerance = 1e-9
val invalidRelevanceScores = scaledCandidates.filter(
  col("relevanceScore").isNull || isnan(col("relevanceScore")) ||
  abs(col("relevanceScore")) === lit(Double.PositiveInfinity) ||
  col("relevanceScore") < -scoreTolerance ||
  col("relevanceScore") > 1.0 + scoreTolerance
).count()

val scalingSummary = scaledCandidates.groupBy("userId").agg(
  min("prediction").as("minimumPrediction"),
  max("prediction").as("maximumPrediction"),
  min("relevanceScore").as("minimumRelevance"),
  max("relevanceScore").as("maximumRelevance")
)
val invalidScalingPools = scalingSummary.filter(
  when(col("minimumPrediction") === col("maximumPrediction"),
    abs(col("minimumRelevance") - 1.0) > scoreTolerance ||
    abs(col("maximumRelevance") - 1.0) > scoreTolerance
  ).otherwise(
    abs(col("minimumRelevance")) > scoreTolerance ||
    abs(col("maximumRelevance") - 1.0) > scoreTolerance
  )
).count()
val constantScorePools = scalingSummary.filter(
  col("minimumPrediction") === col("maximumPrediction")
).count()

val expectedBaseline = scaledCandidates.filter(
  col("recommendationRank") <= recommendationListSize
)
val missingBaselineRows = expectedBaseline.exceptAll(baselineRecommendations).count()
val unexpectedBaselineRows = baselineRecommendations.exceptAll(expectedBaseline).count()

println(s"Invalid scaled scores: $invalidRelevanceScores")
println(s"Pools with incorrect scaling endpoints: $invalidScalingPools")
println(s"Constant-score pools using the all-ones convention: $constantScorePools")
println(s"Baseline rows missing/unexpected: $missingBaselineRows / $unexpectedBaselineRows")
assert(invalidRelevanceScores == 0L, "Scaled relevance scores must be finite and within 0-1.")
assert(invalidScalingPools == 0L, "Every pool must have the expected scaling endpoints.")
assert(missingBaselineRows == 0L && unexpectedBaselineRows == 0L,
  "The baseline must exactly match the first 10 candidates, including scores and ranks.")


Invalid scaled scores: 0


Pools with incorrect scaling endpoints: 0


Constant-score pools using the all-ones convention: 0


Baseline rows missing/unexpected: 0 / 0


scoreTolerance: Double = 1.0E-9
invalidRelevanceScores: Long = 0L
scalingSummary: org.apache.spark.sql.package.DataFrame = [userId: int, minimumPrediction: float ... 3 more fields]
invalidScalingPools: Long = 0L
constantScorePools: Long = 0L
expectedBaseline: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 3 more fields]
missingBaselineRows: Long = 0L
unexpectedBaselineRows: Long = 0L

### Inspect the results

The output schema retains the original predicted rating, per-user rank, and scaled relevance. Aggregate counts describe the full output; the ordered preview shows complete top-10 lists for users 1 and 2. Movie IDs remain the identifiers used by the model. These cells do not load movie titles or evaluate held-out ratings.

The candidate and baseline DataFrames are available during the Spark session. Saved notebook outputs preserve the summaries and previews. The following section exports the complete tables before the session is stopped.


In [27]:
val candidateOutputCount = scaledCandidates.count()
val baselineOutputCount = baselineRecommendations.count()

println(s"Candidate pool: $candidateOutputCount rows across ${candidateUsers.count()} users")
println(s"Baseline recommendations: $baselineOutputCount rows across ${baselineUsers.count()} users")
println("All recommendation validation checks passed.")
baselineRecommendations.printSchema()
baselineRecommendations
  .filter(col("userId").isin(1, 2))
  .orderBy("userId", "recommendationRank")
  .show(20, truncate = false)


Candidate pool: 604000 rows across 6040 users


Baseline recommendations: 60400 rows across 6040 users


All recommendation validation checks passed.


root
 |-- userId: integer (nullable = false)
 |-- movieId: integer (nullable = false)
 |-- prediction: float (nullable = false)
 |-- recommendationRank: integer (nullable = false)
 |-- relevanceScore: double (nullable = true)



+------+-------+----------+------------------+-------------------+
|userId|movieId|prediction|recommendationRank|relevanceScore     |
+------+-------+----------+------------------+-------------------+
|1     |572    |5.4108944 |1                 |1.0                |
|1     |3233   |4.750483  |2                 |0.5065049503327659 |
|1     |527    |4.615465  |3                 |0.40561230125445485|
|1     |318    |4.545293  |4                 |0.35317575979572974|
|1     |953    |4.4822097 |5                 |0.30603660531099836|
|1     |920    |4.439585  |6                 |0.2741852955911288 |
|1     |858    |4.4273562 |7                 |0.26504715162563186|
|1     |989    |4.420023  |8                 |0.2595673295853379 |
|1     |1250   |4.387725  |9                 |0.23543245306038388|
|1     |1851   |4.373576  |10                |0.22485977085870618|
|2     |572    |5.106392  |1                 |1.0                |
|2     |527    |4.445532  |2                 |0.48314517070383

4306   |3                 |0.4812118933848277 |
|2     |318    |4.435184  |4                 |0.4750521824515553 |
|2     |1851   |4.3904037 |5                 |0.4400298046920014 |
|2     |989    |4.2484555 |6                 |0.32901292096050766|
|2     |2762   |4.2468243 |7                 |0.3277371220198571 |
|2     |37     |4.2047377 |8                 |0.29482143476475664|
|2     |1198   |4.192534  |9                 |0.28527699679689067|
|2     |260    |4.1814322 |10                |0.2765944037141002 |
+------+-------+----------+------------------+-------------------+



candidateOutputCount: Long = 604000L
baselineOutputCount: Long = 60400L

## 8. Export baseline outputs

The complete top-100 candidate pool and top-10 baseline lists are saved as Parquet tables under `outputs/als-baseline/<run timestamp>/`. Parquet preserves numeric types and score precision for reuse in Spark or Python. Each table is a directory containing one data part and Spark metadata; share the complete run folder. A new timestamped folder preserves previous exports. Generated outputs are excluded from Git.

The read-back checks compare all rows and columns in both directions, including duplicate multiplicities, to verify that the saved tables exactly match the in-session DataFrames. Consumers should sort by `userId` and `recommendationRank` when displaying lists; file storage does not define a DataFrame's row order.


In [28]:
val repositoryRoot = new java.io.File(ratingsPath).getCanonicalFile
  .getParentFile.getParentFile.getParentFile
val exportRunId = java.time.Instant.now().toString.replace(":", "-")
val exportDirectory = new java.io.File(
  repositoryRoot, s"outputs/als-baseline/$exportRunId"
).getPath
val candidatesExportPath = s"$exportDirectory/top-100-candidates"
val baselineExportPath = s"$exportDirectory/top-10-baseline"

scaledCandidates.coalesce(1).write
  .mode("errorifexists").parquet(candidatesExportPath)
baselineRecommendations.coalesce(1).write
  .mode("errorifexists").parquet(baselineExportPath)

println(s"Candidate export: $candidatesExportPath")
println(s"Baseline export: $baselineExportPath")


Candidate export: /Users/darylshy/KSU/Fall 26/Big Data Analytics/Group Project/history-aware-popularity-calibration/outputs/als-baseline/2026-10-10T23-06-55.392541Z/top-100-candidates


Baseline export: /Users/darylshy/KSU/Fall 26/Big Data Analytics/Group Project/history-aware-popularity-calibration/outputs/als-baseline/2026-10-10T23-06-55.392541Z/top-10-baseline


repositoryRoot: java.io.File = /Users/darylshy/KSU/Fall 26/Big Data Analytics/Group Project/history-aware-popularity-calibration
exportRunId: String = "2026-10-10T23-06-55.392541Z"
exportDirectory: String = "/Users/darylshy/KSU/Fall 26/Big Data Analytics/Group Project/history-aware-popularity-calibration/outputs/als-baseline/2026-10-10T23-06-55.392541Z"
candidatesExportPath: String = "/Users/darylshy/KSU/Fall 26/Big Data Analytics/Group Project/history-aware-popularity-calibration/outputs/als-baseline/2026-10-10T23-06-55.392541Z/top-100-candidates"
baselineExportPath: String = "/Users/darylshy/KSU/Fall 26/Big Data Analytics/Group Project/history-aware-popularity-calibration/outputs/als-baseline/2026-10-10T23-06-55.392541Z/top-10-baseline"

In [29]:
val exportedCandidates = spark.read.parquet(candidatesExportPath)
  .select("userId", "movieId", "prediction", "recommendationRank", "relevanceScore")
val exportedBaseline = spark.read.parquet(baselineExportPath)
  .select("userId", "movieId", "prediction", "recommendationRank", "relevanceScore")

val missingExportedCandidates = scaledCandidates.exceptAll(exportedCandidates).count()
val unexpectedExportedCandidates = exportedCandidates.exceptAll(scaledCandidates).count()
val missingExportedBaseline = baselineRecommendations.exceptAll(exportedBaseline).count()
val unexpectedExportedBaseline = exportedBaseline.exceptAll(baselineRecommendations).count()

println(s"Candidate export rows missing/unexpected: $missingExportedCandidates / $unexpectedExportedCandidates")
println(s"Baseline export rows missing/unexpected: $missingExportedBaseline / $unexpectedExportedBaseline")
assert(missingExportedCandidates == 0L && unexpectedExportedCandidates == 0L,
  "Exported candidates must exactly match the generated candidate pool.")
assert(missingExportedBaseline == 0L && unexpectedExportedBaseline == 0L,
  "Exported baseline recommendations must exactly match the generated lists.")

println(s"Verified candidate export: ${exportedCandidates.count()} rows")
println(s"Verified baseline export: ${exportedBaseline.count()} rows")


Candidate export rows missing/unexpected: 0 / 0


Baseline export rows missing/unexpected: 0 / 0


Verified candidate export: 604000 rows


Verified baseline export: 60400 rows


exportedCandidates: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 3 more fields]
exportedBaseline: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 3 more fields]
missingExportedCandidates: Long = 0L
unexpectedExportedCandidates: Long = 0L
missingExportedBaseline: Long = 0L
unexpectedExportedBaseline: Long = 0L

## 9. Stop the Spark session

`spark.stop()` shuts down the Spark session and its underlying Spark context, releasing their resources. Run this cell after data inspection and any subsequent Spark work. Further Spark operations require initializing a session again.

The recommendation caches are explicitly released before Spark is stopped.


In [30]:
scoredCandidates.unpersist(blocking = true)
scaledCandidates.unpersist(blocking = true)

spark.stop()
println("Spark session stopped.")


Spark session stopped.


res30_0: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 1 more field]
res30_1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 3 more fields]